An example of schema evolution error that was correctly handled by the AutoLoader (Encountered unknown fields during parsing: {"bidding_zone":"ES"}, **which can be fixed by an automatic retry: true**)
![image_1784141485879.png](./image_1784141485879.png "image_1784141485879.png")

After re-run the files from ES bidding zone are correctly ingested to bronze layer:

In [0]:
%sql
SELECT *
FROM dbr_dev.gabrielajaniszews786_bronze.entsoe_prices
WHERE bidding_zone = 'ES'
LIMIT 10

### Experimenting with different trigger types

**Streaming stats — Auto Loader (processingTime trigger)**
The stream runs as micro-batches on a fixed trigger interval, so both charts are spiky by design rather than flat. Processing rate sits consistently above input rate, which means Spark is draining data faster than it arrives — the consumer is keeping up and catching up on backlog, so no lag accumulates.
The processing-rate spikes line up with bursts of new files from the producer: the fetch job writes files in batches, Auto Loader discovers a chunk at once, and each trigger crunches the accumulated files in a single burst before going quiet until the next batch. The large initial spike is the startup backlog being cleared in the first batch.
Batch duration follows the same pattern, rising on heavier batches and settling on lighter ones. As long as batch duration stays under the trigger interval, batches don't queue up and the stream stays stable. The input-below-processing relationship is the healthy signal here; the inverse would indicate backpressure and a growing backlog.

![image_1784143512523.png](./image_1784143512523.png "image_1784143512523.png")



**Note — trigger(once=True) reporting no new data**
Ran a one-time load with trigger(once=True) and it kept starting and stopping instantly: status = Stopped, isDataAvailable = False, lastProgress = None, no exception. Files were clearly present in landing and one zone (LT) was missing from the target table, so the "nothing to do" result was misleading.

Root cause was a leftover continuous stream (processingTime) still active on the same checkpointLocation. Two queries sharing one checkpoint conflict: the background stream owned/advanced the checkpoint state, so the once query saw everything as already processed and exited without a batch. Once I stopped the active queries (spark.streams.active → stop()), the next run picked up only the outstanding LT files and left the existing rows untouched — clean incremental load, no full reload needed.

**Takeaway**: always stop running queries before launching another against the same checkpoint, and check spark.streams.active when a trigger reports no data even though new files exist. isDataAvailable: False means "checkpoint thinks it's caught up," not "no files on disk."

### Adding a broken file to landing to observe _rescued_data column


In [0]:
# Type mismatch -> value lands in _rescued_data
CATALOG    = "dbr_dev"
SCHEMA     = "gabrielajaniszews786_bronze"
LANDING    = f"/Volumes/{CATALOG}/{SCHEMA}/entsoe_landing"
bad = '{"timestamp_utc":"2026-07-10T10:00:00Z","price":"138.22","currency":"EUR","unit":"MWh","bidding_zone":"PL"}\n'
with open(f"{LANDING}/prices_PL_BROKEN_test.json", "w") as f:
    f.write(bad)

In [0]:
%sql

--Checking the _rescued_data column --
SELECT source_file, _rescued_data
FROM dbr_dev.gabrielajaniszews786_bronze.entsoe_prices
WHERE _rescued_data IS NOT NULL

The data was correctly processed and the price data in a string format was rescued in the _rescued_data column.

In [0]:
# Removing the broken file from landing
dbutils.fs.rm(f"{LANDING}/prices_PL_BROKEN_test.json")


## Scheduled streaming job — design & cost awareness
The pipeline is split into two decoupled jobs. The producer job (event_hub_task) simulates the external live source (data-center meters pushing to Event Hub) — in production this is an outside system, so it is not part of the ingestion pipeline; here it run it manually to generate the feed. The consumer job is the actual pipeline and is the one on a schedule (every 15 min, see screenshots).
The consumer uses trigger(availableNow=True) rather than a continuous stream: it drains the events accumulated since the last run in incremental micro-batches, then stops, so the job cluster auto-terminates between runs. This is the main cost lever — we pay for minutes per run instead of a 24/7 cluster. Data-center telemetry does not need sub-second latency, so periodic incremental drain is the right freshness/cost trade-off.
Incremental + exactly-once: the checkpoint tracks Event Hub offsets, so each run reads only new events (no duplicates, no reprocessing), and the write to Delta is transactional — the pipeline recovers from failure without loss or duplication. A data-quality gate at the end (null keys, duplicate event_id, out-of-range values) raises on bad data and fails the job task. ![image_1784623983051.png](./image_1784623983051.png "image_1784623983051.png")